# Gemini 3.1 Flash 图片生成 — Google GenAI SDK

本 Notebook 演示如何通过 **七牛 AIToken** 平台，使用 **Google GenAI SDK** (`google-genai`) 调用 **Gemini 3.1 Flash Image Preview** 模型生成图片。

## 模型简介

Gemini 3.1 Flash Image Preview 是 Google 推出的轻量级多模态图像生成模型，响应速度快、生成质量高。

- **文本生成图片**：根据自然语言描述生成高质量图片
- **图片编辑（图生图）**：基于输入图片进行编辑、风格转换等操作，支持单图和多图输入
- **自定义宽高比**：支持 `1:1`、`2:3`、`3:2`、`3:4`、`4:3`、`4:5`、`5:4`、`9:16`、`16:9`、`21:9` 等多种比例
- **多种分辨率**：支持 `512`、`1K`、`2K`、`4K`

**模型名称**：`gemini-3.1-flash-image-preview`

## 1. 环境准备

安装 `google-genai` 和 `Pillow` 依赖包，并配置客户端指向七牛 AIToken 平台。

> **前置条件**：需要 Python 3.9 或更高版本，并设置环境变量 `QINIU_API_KEY`。

In [ ]:
# 安装 Google GenAI SDK 和 Pillow（用于图片处理）
%pip install google-genai Pillow -q

In [ ]:
import os
import base64
import urllib.request
from google import genai
from google.genai import types
from google.genai.types import HttpOptions
from IPython.display import Image, display

# ==================== 配置参数 ====================
MODEL_ID = "gemini-3.1-flash-image-preview"  # 模型 ID
BASE_URL = "https://api.qnaigc.com/bypass/vertex"  # 七牛 AIToken 国内节点
# ==================================================

# 从环境变量读取 API Key
api_key = os.environ.get("QINIU_API_KEY", "<your-api-key>")

# 初始化客户端
# 使用 vertexai=True 并通过 HttpOptions 配置自定义端点和认证
client = genai.Client(
    vertexai=True,
    http_options=HttpOptions(
        api_version="v1",
        base_url=BASE_URL,
        headers={"Authorization": f"Bearer {api_key}"},
    ),
)

print("环境配置完成!")
print(f"  API 端点: {BASE_URL}")
print(f"  模型: {MODEL_ID}")

In [ ]:
def display_response(response):
    """从响应中提取文本和图片并展示，返回第一张图片的 bytes 数据"""
    image_data = None
    for part in response.candidates[0].content.parts:
        # 跳过思考过程的 part
        if part.thought:
            continue
        if part.text is not None:
            print(part.text)
        elif part.inline_data is not None:
            # inline_data.data 已经是原始字节，无需 base64 解码
            raw = part.inline_data.data
            display(Image(data=raw))
            if image_data is None:
                image_data = raw
    return image_data

## 2. 基础用法 — 文本生成图片

通过 `generate_content()` 方法，设置 `response_modalities=["TEXT", "IMAGE"]` 使模型返回图片。图片数据以 `inline_data`（base64 编码）的形式包含在响应的 `parts` 中。

In [ ]:
# 基础调用示例：生成一只可爱的橘猫
response = client.models.generate_content(
    model=MODEL_ID,
    contents="画一只可爱的橘猫",
    config=types.GenerateContentConfig(
        response_modalities=["TEXT", "IMAGE"],
    ),
)

# 展示生成的图片
image_data = display_response(response)

## 3. 自定义宽高比和分辨率

通过 `image_config` 参数可以控制生成图片的宽高比和分辨率。

### 支持的宽高比
`1:1`、`2:3`、`3:2`、`3:4`、`4:3`、`4:5`、`5:4`、`9:16`、`16:9`、`21:9`

### 支持的分辨率
`512`（仅 3.1 Flash）、`1K`、`2K`、`4K`

In [ ]:
# 示例：生成风景图片（16:9 宽屏，4K 分辨率）
response = client.models.generate_content(
    model=MODEL_ID,
    contents="画一幅宁静的日式庭院，樱花树下有一条石板小路，远处是朦胧的山峦，水彩画风格",
    config=types.GenerateContentConfig(
        response_modalities=["TEXT", "IMAGE"],
        image_config=types.ImageConfig(
            aspect_ratio="16:9",
            image_size="4K",
        ),
    ),
)

image_data = display_response(response)

In [ ]:
# 示例：生成竖版海报（9:16 比例）
response = client.models.generate_content(
    model=MODEL_ID,
    contents="设计一张赛博朋克风格的城市夜景海报，霓虹灯闪烁，雨中的街道倒映着五彩灯光",
    config=types.GenerateContentConfig(
        response_modalities=["TEXT", "IMAGE"],
        image_config=types.ImageConfig(
            aspect_ratio="9:16",
            image_size="4K",
        ),
    ),
)

image_data = display_response(response)

## 4. 图生图 — 单图编辑

通过在 `contents` 中同时传入文本指令和图片数据，可以基于已有图片进行编辑。

GenAI SDK 支持通过 `Part.from_bytes()` 传入图片的原始字节数据。

In [ ]:
# 下载示例图片
image_url = "https://aitoken-public.qnaigc.com/example/generate-image/image-to-image-1.jpg"
image_bytes = urllib.request.urlopen(image_url).read()

# 展示原始图片
print("原始图片:")
display(Image(data=image_bytes))

In [ ]:
# 示例：单图编辑 — 将图片颜色改为红色
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        types.Content(
            role="user",
            parts=[
                types.Part.from_text(text="Change this image to red."),
                types.Part.from_bytes(data=image_bytes, mime_type="image/jpeg"),
            ],
        )
    ],
    config=types.GenerateContentConfig(
        response_modalities=["TEXT", "IMAGE"],
    ),
)

print("编辑后的图片:")
image_data = display_response(response)

## 5. 图生图 — 多图编辑

支持在同一请求中传入多张图片，模型会综合理解所有图片内容并根据文本指令进行处理。常见场景包括：图片融合、风格迁移、多图对比编辑等。

In [ ]:
# 下载第二张示例图片
image_url_2 = "https://aitoken-public.qnaigc.com/example/generate-video/running-man.jpg"
image_bytes_2 = urllib.request.urlopen(image_url_2).read()

# 展示两张原始图片
print("图片 1:")
display(Image(data=image_bytes))
print("图片 2:")
display(Image(data=image_bytes_2))

In [ ]:
# 示例：多图编辑 — 将两张图片融合为一张
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        types.Content(
            role="user",
            parts=[
                types.Part.from_text(
                    text="Merge these two images into one, combining the elements of both images harmoniously."
                ),
                types.Part.from_bytes(data=image_bytes, mime_type="image/jpeg"),
                types.Part.from_bytes(data=image_bytes_2, mime_type="image/jpeg"),
            ],
        )
    ],
    config=types.GenerateContentConfig(
        response_modalities=["TEXT", "IMAGE"],
    ),
)

print("融合后的图片:")
image_data = display_response(response)

## 6. 保存图片到本地

In [ ]:
# 将生成的图片保存到本地
if image_data:
    output_path = "output.png"
    with open(output_path, "wb") as f:
        f.write(image_data)
    print(f"图片已保存到: {output_path}")

## 7. 参数说明

### 客户端配置

| 参数 | 说明 |
|------|------|
| `vertexai` | 设为 `True`，使用 Vertex AI 协议 |
| `http_options.api_version` | API 版本，固定为 `"v1"` |
| `http_options.base_url` | API 端点地址，设为 `https://api.qnaigc.com/bypass/vertex` |
| `http_options.headers` | 请求头，需包含 `Authorization: Bearer <API_KEY>` |

### GenerateContentConfig 主要参数

| 参数 | 类型 | 说明 |
|------|------|------|
| `response_modalities` | list[str] | 响应模态，图片生成需包含 `"IMAGE"`，如 `["TEXT", "IMAGE"]` |
| `image_config` | ImageConfig | 图像配置，控制宽高比和分辨率 |
| `temperature` | float | 生成温度，范围 0.0-2.0，越高越随机 |
| `top_p` | float | Top-P 采样，范围 0.0-1.0 |
| `max_output_tokens` | int | 最大输出 token 数 |

### ImageConfig 参数

| 参数 | 类型 | 说明 |
|------|------|------|
| `aspect_ratio` | string | 图像宽高比，可选值：`1:1`、`2:3`、`3:2`、`3:4`、`4:3`、`4:5`、`5:4`、`9:16`、`16:9`、`21:9` |
| `image_size` | string | 图像分辨率，可选值：`512`、`1K`、`2K`、`4K` |

### 环境变量

| 环境变量 | 说明 |
|----------|------|
| `QINIU_API_KEY` | 七牛 AIToken 平台的 API Key |

## 更多资源

- [七牛 AIToken 文档](https://developer.qiniu.com/aitokenapi)
- [Google GenAI SDK 文档](https://googleapis.github.io/python-genai/)
- [Gemini 图片生成文档](https://ai.google.dev/gemini-api/docs/image-generation)
- [七牛 API 调用示例文档](https://apidocs.qnaigc.com)